<a href="https://colab.research.google.com/github/davesagit123/blank-app/blob/main/racenet_sectionals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# CUSTOM MODIFIED THOMPSON TAU IMPLEMENTATION
# =============================================================================

def thompson_tau_test(data, alpha=0.01):
    """
    A standalone implementation of the Modified Thompson Tau test.
    Returns the indices of outliers.
    """
    n = len(data)
    if n < 3:
        return []

    indices = list(range(n))
    outliers = []

    # We iterate to find multiple outliers (recursive/iterative approach)
    temp_data = data.copy().values
    temp_indices = list(range(n))

    while True:
        n_curr = len(temp_data)
        if n_curr < 3: break

        mean = np.mean(temp_data)
        std = np.std(temp_data, ddof=1)
        if std == 0: break

        # Student's t critical value
        t_crit = stats.t.ppf(1 - alpha / 2, n_curr - 2)

        # Thompson Tau value
        tau = (t_crit * (n_curr - 1)) / (np.sqrt(n_curr) * np.sqrt(n_curr - 2 + t_crit**2))

        # Find the value furthest from the mean
        deviations = np.abs(temp_data - mean)
        max_dev_idx = np.argmax(deviations)

        if deviations[max_dev_idx] > (tau * std):
            # It is an outlier
            outliers.append(temp_indices.pop(max_dev_idx))
            temp_data = np.delete(temp_data, max_dev_idx)
        else:
            break

    return outliers

# =============================================================================
# 2. DEFINE YOUR DATA
# =============================================================================

data = {
    'Runner': ['Ceolwulf', 'Gringotts', 'Lindermann', 'Midnight Dynamite', 'Green Spaces',
               'Autumn Glow', 'Aeliana', 'Fangirl', 'Idle Flyer', 'Lady Shenandoah', 'Sheza Alibi'],
    'L3_800': [-3.06, -3.06, -2.87, -3.03, -2.95, -3.43, -4.36, -3.24, -2.88, -2.65, -3.25],
    'L3_600': [-2.42, -2.74, -1.91, -2.68, -2.25, -2.90, -3.43, -2.87, -2.43, -2.23, -2.58],
    'L3_400': [-1.53, -2.13, -1.02, -1.85, -1.54, -2.06, -2.26, -2.10, -1.64, -1.56, -1.99],
    'L3_200': [-0.44, -0.89, -0.14, -0.79, -0.52, -0.79, -0.98, -0.86, -0.64, -0.59, -0.95],
    'L3_Pace': [58, 60, 80, 53, 65, 71, 73, 60, 57, 67, 82],
    'L10_800': [-3.31, -2.85, -2.31, -2.32, -2.02, -3.03, -3.63, -3.27, -2.62, -2.45, -2.62],
    'L10_600': [-2.78, -2.64, -1.81, -1.95, -1.74, -2.71, -3.09, -2.86, -2.27, -2.12, -2.26],
    'L10_400': [-1.91, -2.12, -1.02, -1.44, -1.24, -2.06, -2.08, -2.08, -1.67, -1.57, -1.62],
    'L10_200': [-0.64, -0.83, -0.20, -0.52, -0.50, -0.87, -0.92, -0.83, -0.73, -0.71, -0.74],
    'L10_Pace': [66, 60, 75, 60, 44, 62, 46, 59, 58, 69, 56],
    'MissedStart': [False, False, False, False, False, False, True, False, False, False, False],
    'ClassRise': [True, True, True, True, True, True, True, True, True, True, True]
}

df = pd.DataFrame(data)

def find_positive_outliers(data_series, alpha=0.01):
    outlier_indices = thompson_tau_test(data_series, alpha=alpha)
    if len(outlier_indices) > 0:
        outlier_values = data_series.iloc[outlier_indices]
        min_value = data_series.min()
        # Identify if outliers are the 'fast' ones (lower values)
        positive_outliers = [idx for idx in outlier_indices if data_series.iloc[idx] <= min_value + 0.5]
        return positive_outliers
    return []

# Run test
critical_split = 'L3_600'
outlier_idx = find_positive_outliers(df[critical_split], alpha=0.01)
df['Tau_Outlier'] = False
if outlier_idx:
    df.loc[outlier_idx, 'Tau_Outlier'] = True

# Engineering features
df['Trajectory'] = df['L3_600'] - df['L10_600']
df['FastPace'] = df['L3_Pace'] > 70
df['Consistency'] = abs(df['L3_600'] - df['L10_600'])

# Weights
WEIGHT_TAU, WEIGHT_TRAJECTORY, WEIGHT_PACE, WEIGHT_CONSISTENCY = 4, 3, 2, 1

def calculate_score(row):
    score = 0
    if row['Tau_Outlier']: score += WEIGHT_TAU
    if row['Trajectory'] < -0.1: score += WEIGHT_TRAJECTORY
    if row['FastPace'] and row['L3_600'] < -2.5: score += WEIGHT_PACE
    if row['Consistency'] < 0.3: score += WEIGHT_CONSISTENCY
    if row['MissedStart']: score += 1
    if row['ClassRise'] and row['Trajectory'] < -0.1: score += 1
    return score

df['Score'] = df.apply(calculate_score, axis=1)
df = df.sort_values('Score', ascending=False)

display_cols = ['Runner', 'Score', 'Tau_Outlier', 'L3_600', 'L10_600', 'Trajectory', 'L3_Pace', 'FastPace']
print(tabulate(df[display_cols], headers='keys', tablefmt='grid', floatfmt=".2f"))

print("\nBetting Recommendation:", df[df['Tau_Outlier']]['Runner'].tolist() if not df[df['Tau_Outlier']].empty else "No clear standout")

+----+-------------------+---------+---------------+----------+-----------+--------------+-----------+------------+
|    | Runner            |   Score | Tau_Outlier   |   L3_600 |   L10_600 |   Trajectory |   L3_Pace | FastPace   |
+====+===================+=========+===============+==========+===========+==============+===========+============+
|  6 | Aeliana           |       7 | False         |    -3.43 |     -3.09 |        -0.34 |        73 | True       |
+----+-------------------+---------+---------------+----------+-----------+--------------+-----------+------------+
|  5 | Autumn Glow       |       7 | False         |    -2.90 |     -2.71 |        -0.19 |        71 | True       |
+----+-------------------+---------+---------------+----------+-----------+--------------+-----------+------------+
| 10 | Sheza Alibi       |       6 | False         |    -2.58 |     -2.26 |        -0.32 |        82 | True       |
+----+-------------------+---------+---------------+----------+---------

new version

In [13]:
import pandas as pd
import numpy as np
from scipy import stats
from tabulate import tabulate
import io
import re
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# 1. PARSING UTILITY
# =============================================================================

def parse_race_data(raw_text):
    """Parses the raw tabular text, cleaning 'L' suffixes and extracting runners."""
    lines = [l.strip() for l in raw_text.strip().split('\n') if l.strip()]
    data_lines = []
    for line in lines:
        if 'Runner Time' in line or 'Runner' in line and 'L800m' in line:
            continue
        cleaned = line.replace('L', '')
        parts = re.split(r'\t+| {2,}', cleaned)
        if len(parts) >= 7:
            data_lines.append(parts[:7])

    columns = ['Runner', 'RunnerTime', 'EarlyPace', '800m', '600m', '400m', '200m']
    df = pd.DataFrame(data_lines, columns=columns)
    for col in columns[1:]:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# =============================================================================
# 2. RAW DATA INPUT
# =============================================================================

raw_l3_data = """
1. Ceolwulf	-13.96L	58	-3.06L	-2.42L	-1.53L	-0.44L
2. Gringotts	-16.56L	60	-3.06L	-2.74L	-2.13L	-0.89L
3. Lindermann	-19.44L	80	-2.87L	-1.91L	-1.02L	-0.14L
4. Midnight Dynamite	-13.07L	53	-3.03L	-2.68L	-1.85L	-0.79L
7. Green Spaces	-15.00L	65	-2.95L	-2.25L	-1.54L	-0.52L
8. Autumn Glow	-22.24L	71	-3.43L	-2.90L	-2.06L	-0.79L
9. Aeliana	-26.71L	73	-4.36L	-3.43L	-2.26L	-0.98L
10. Fangirl	-16.30L	60	-3.24L	-2.87L	-2.10L	-0.86L
11. Idle Flyer	-16.11L	57	-2.88L	-2.43L	-1.64L	-0.64L
12. Lady Shenandoah	-17.32L	67	-2.65L	-2.23L	-1.56L	-0.59L
14. Sheza Alibi	-21.16L	82	-3.25L	-2.58L	-1.99L	-0.95L


"""

raw_l10_data = """
1. Ceolwulf	-17.82L	66	-3.31L	-2.78L	-1.91L	-0.64L
2. Gringotts	-15.37L	60	-2.85L	-2.64L	-2.12L	-0.83L
3. Lindermann	-16.11L	75	-2.31L	-1.81L	-1.02L	-0.20L
4. Midnight Dynamite	-10.48L	60	-2.32L	-1.95L	-1.44L	-0.52L
7. Green Spaces	-6.39L	44	-2.02L	-1.74L	-1.24L	-0.50L
8. Autumn Glow	-17.49L	62	-3.03L	-2.71L	-2.06L	-0.87L
9. Aeliana	-12.57L	46	-3.63L	-3.09L	-2.08L	-0.92L
10. Fangirl	-16.06L	59	-3.27L	-2.86L	-2.08L	-0.83L
11. Idle Flyer	-13.44L	58	-2.62L	-2.27L	-1.67L	-0.73L
12. Lady Shenandoah	-15.30L	69	-2.45L	-2.12L	-1.57L	-0.71L
14. Sheza Alibi	-11.47L	56	-2.62L	-2.26L	-1.62L	-0.74L


"""

df_l3 = parse_race_data(raw_l3_data)
df_l10 = parse_race_data(raw_l10_data)
df = df_l3.copy().rename(columns={'800m': 'L3_800', '600m': 'L3_600', '400m': 'L3_400', '200m': 'L3_200', 'EarlyPace': 'L3_Pace'})
df = pd.merge(df, df_l10[['Runner', '600m']].rename(columns={'600m': 'L10_600'}), on='Runner')

# =============================================================================
# 3. ANALYSIS LOGIC
# =============================================================================

def modified_thompson_tau(data, alpha=0.05):
    data_array = np.array(data)
    n = len(data_array)
    if n < 3: return []
    remaining_indices = list(range(n))
    outlier_indices = []
    while len(remaining_indices) >= 3:
        curr = data_array[remaining_indices]
        mean, std = np.mean(curr), np.std(curr, ddof=1)
        if std == 0: break
        t_crit = stats.t.ppf(1 - alpha/(2*len(curr)), len(curr)-2)
        tau = (t_crit * (len(curr)-1)) / (np.sqrt(len(curr)) * np.sqrt(len(curr)-2 + t_crit**2))
        dev = np.abs(curr - mean)
        max_idx = np.argmax(dev)
        if dev[max_idx] / std > tau:
            outlier_indices.append(remaining_indices.pop(max_idx))
        else: break
    return outlier_indices

outlier_idx = modified_thompson_tau(df['L3_600'], alpha=0.10)
df['Tau_Outlier'] = False
if outlier_idx: df.loc[outlier_idx, 'Tau_Outlier'] = True

df['Trajectory'] = df['L3_600'] - df['L10_600']
df['FastPace'] = df['L3_Pace'] > 70
df['Score'] = 0

for idx, row in df.iterrows():
    s = 0
    if row['Tau_Outlier'] and row['L3_600'] < df['L3_600'].mean(): s += 4
    if row['Trajectory'] < -0.1: s += 3
    if row['FastPace'] and row['L3_600'] < -2.5: s += 2
    if abs(row['Trajectory']) < 0.25: s += 1
    if row['L3_200'] < -0.85: s += 1
    df.at[idx, 'Score'] = s

df = df.sort_values('Score', ascending=False)
# Added Tau_Outlier back to display list
display_cols = ['Runner', 'Score', 'Tau_Outlier', 'L3_600', 'L10_600', 'Trajectory', 'L3_Pace']
print(tabulate(df[display_cols], headers='keys', tablefmt='grid', floatfmt=".2f"))

standouts = df[df['Tau_Outlier'] == True]['Runner'].tolist()
print(f"\nStatistical Outliers: {standouts if standouts else 'None'}")
print("Betting Focus:", df[df['Score'] >= 6]['Runner'].tolist())

+----+----------------------+---------+---------------+----------+-----------+--------------+-----------+
|    | Runner               |   Score | Tau_Outlier   |   L3_600 |   L10_600 |   Trajectory |   L3_Pace |
+====+======================+=========+===============+==========+===========+==============+===========+
| 10 | 14. Sheza Alibi      |       6 | False         |    -2.58 |     -2.26 |        -0.32 |        82 |
+----+----------------------+---------+---------------+----------+-----------+--------------+-----------+
|  5 | 8. Autumn Glow       |       6 | False         |    -2.90 |     -2.71 |        -0.19 |        71 |
+----+----------------------+---------+---------------+----------+-----------+--------------+-----------+
|  6 | 9. Aeliana           |       6 | False         |    -3.43 |     -3.09 |        -0.34 |        73 |
+----+----------------------+---------+---------------+----------+-----------+--------------+-----------+
|  1 | 2. Gringotts         |       5 | False 